In [0]:
%run ./01_config

In [0]:
"""
13_uncertainty.py  —  Statistical uncertainty quantification

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 15.
"""

# 13 — Statistical uncertainty

# Every headline in this study has so far been a point estimate. This notebook attaches
# intervals to them, which changes how several results should be read: a criterion met at
# the point estimate is not the same as a criterion met with confidence.

# Quantity  -  Method
# Concordance, per model  -  Non-parametric bootstrap over hold-out intervals (1,000 resamples)
# Escalation-model AUC  -  Bootstrap over hold-out defect notifications
# Weibull parameter recovery  -  Bootstrap over each class's training intervals
# Backtest regime deltas  -  Paired across replications, with Wilcoxon signed-rank

# The bootstrap resamples intervals rather than equipment items, which understates
# uncertainty where one item contributes several intervals. A cluster bootstrap over
# equipment is also computed for the headline concordance so the two can be compared.

# Requires (run before this notebook): pip install --quiet lifelines
# (interpreter restart required after installation)

# Shared configuration from '01_config' is assumed to be in scope.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from lifelines import CoxPHFitter

use_project_schema()

INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e0e0e0"
ACCENT, WARM, GREEN = "#2b6cb0", "#c05621", "#2f855a"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG = f"{FIGURES}/{DATASET_VERSION}_"
B_BOOT = 1000
RNG = np.random.default_rng(42)

def emit(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIG}{name}.svg", bbox_inches="tight", format="svg")
    print(f"saved {FIG}{name}.png / .svg")
    plt.show(); plt.close(fig)

def concordance(time, event, risk):
    time, event, risk = np.asarray(time, float), np.asarray(event), np.asarray(risk, float)
    conc = perm = 0.0
    for i in np.where(event == 1)[0]:
        m = time > time[i]
        if not m.any():
            continue
        conc += np.sum(risk[i] > risk[m]) + 0.5 * np.sum(risk[i] == risk[m])
        perm += m.sum()
    return conc / perm if perm else np.nan

def pct_ci(v, lo=2.5, hi=97.5):
    v = np.asarray([x for x in v if x == x])
    return float(np.percentile(v, lo)), float(np.percentile(v, hi)), float(v.std(ddof=1))

# Rebuild the hold-out risk scores

# The prediction layer is refitted here rather than reloaded, because the bootstrap needs
# the per-interval risk scores, which notebook 09 does not persist. The fitted concordance
# is checked against the stored value as a reproduction test before any interval is quoted.

RHO = float(spark.table(tbl("results_rho_selection")).toPandas()
            .sort_values("holdout_loglik").iloc[-1]["rho"])

iv = spark.table(tbl("gold_survival_intervals")).toPandas()
iv["interval_start"] = pd.to_datetime(iv.interval_start)
eq = spark.table(tbl("silver_equipment")).select(
    "equipment_id", "manufacturer", "planning_plant").toPandas()
iv = iv.merge(eq, on="equipment_id", how="left")
iv["terminated_by_pm"] = (iv.termination == "CENSORED_PM").astype(int)
iv = iv.sort_values(["equipment_id", "seq_no"]).reset_index(drop=True)

def virtual_age(df, rho):
    out = np.zeros(len(df))
    dur = df.duration_days.values.astype(float)
    evt = df.event_observed.values
    pm = df.terminated_by_pm.values
    for _, idx in df.groupby("equipment_id").indices.items():
        va = 0.0
        for i in idx:
            out[i] = va
            va = 0.0 if evt[i] == 1 else ((va + dur[i]) * (1 - rho) if pm[i] == 1 else va + dur[i])
    return out

iv["va"] = virtual_age(iv, RHO)
levels = {"manufacturer": sorted(iv.manufacturer.dropna().unique())[1:],
          "plant": sorted(iv.planning_plant.dropna().unique())[1:],
          "equipment_class": sorted(iv.equipment_class.unique())[1:]}

def design(df):
    X = pd.DataFrame(index=df.index)
    for m in levels["manufacturer"]:
        X[f"mfr_{m}"] = (df.manufacturer == m).astype(float)
    for p in levels["plant"]:
        X[f"plant_{p}"] = (df.planning_plant == p).astype(float)
    for c in levels["equipment_class"]:
        X[f"cls_{c}"] = (df.equipment_class == c).astype(float)
    X["criticality"] = df.criticality.fillna(2.5).astype(float) - 2.5
    X["virtual_age"] = df.va.values / 500.0
    X["prior_failures"] = df.prior_failure_count.astype(float)
    return X

def weibull_aft(X, t, e, v):
    X = np.asarray(X, float); k = X.shape[1]
    def nll(p):
        b, eta = np.exp(p[0]), np.exp(p[1])
        ei = eta * np.exp(-(X @ p[2:]) / b)
        x1, x0 = (v + t) / ei, v / ei
        logh = np.log(b / ei) + (b - 1) * np.log(np.maximum(x1, 1e-9))
        return -(np.sum(e * logh) - np.sum(x1 ** b - x0 ** b))
    r = minimize(nll, np.r_[np.log(1.8), np.log(max(t.mean(), 1) * 2), np.zeros(k)],
                 method="L-BFGS-B", options={"maxiter": 4000})
    return float(np.exp(r.x[0])), float(np.exp(r.x[1])), r.x[2:]

tr = iv[iv.interval_start <= pd.Timestamp(TRAIN_CUTOFF)].reset_index(drop=True)
te = iv[iv.interval_start > pd.Timestamp(TRAIN_CUTOFF)].reset_index(drop=True)
Xtr, Xte = design(tr), design(te).reindex(columns=design(tr).columns, fill_value=0.0)
ttr, etr, vtr = (tr.duration_days.values.astype(float),
                 tr.event_observed.values.astype(float), tr.va.values)
tte, ete = te.duration_days.values.astype(float), te.event_observed.values.astype(float)

risk = {}
b_pool, eta_pool, coefs = weibull_aft(Xtr.values, ttr, etr, vtr)
risk["Weibull AFT"] = Xte.values @ coefs
dtr = Xtr.copy(); dtr["T"] = ttr; dtr["E"] = etr
cph = CoxPHFitter(penalizer=0.05).fit(dtr, "T", "E")
risk["Cox PH"] = cph.predict_partial_hazard(Xte).values
risk["Weibull (virtual age only)"] = (te.va.values / eta_pool) ** b_pool

stored = spark.table(tbl("results_concordance")).toPandas()
stored_cox = float(stored[(stored.model == "Cox PH") &
                          (stored.equipment_class == "— fleet —")].concordance.iloc[0])
refit_cox = concordance(tte, ete, risk["Cox PH"])
print(f"stored Cox concordance {stored_cox:.4f} | refit {refit_cox:.4f} | "
      f"delta {abs(stored_cox - refit_cox):.4f}")
assert abs(stored_cox - refit_cox) < 0.01, "refit does not reproduce the stored result"

# Bootstrap intervals on concordance

rows = []
for model, r in risk.items():
    obs = concordance(tte, ete, r)
    boots = []
    rs = np.random.default_rng(7)
    n = len(tte)
    for _ in range(B_BOOT):
        idx = rs.integers(0, n, n)
        boots.append(concordance(tte[idx], ete[idx], r[idx]))
    lo, hi, se = pct_ci(boots)
    rows.append({"quantity": f"concordance — {model}", "estimate": round(obs, 4),
                 "ci_low": round(lo, 4), "ci_high": round(hi, 4), "se": round(se, 4),
                 "method": "interval bootstrap", "n_resamples": B_BOOT})
    print(f"{model:28s} {obs:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  se {se:.4f}")

# cluster bootstrap over equipment, for the headline model
items = te.equipment_id.values
uniq = np.unique(items)
idx_by_item = {u: np.where(items == u)[0] for u in uniq}
boots = []
rs = np.random.default_rng(11)
for _ in range(B_BOOT):
    pick = rs.choice(uniq, len(uniq), replace=True)
    idx = np.concatenate([idx_by_item[u] for u in pick])
    boots.append(concordance(tte[idx], ete[idx], risk["Cox PH"][idx]))
lo, hi, se = pct_ci(boots)
rows.append({"quantity": "concordance — Cox PH (cluster bootstrap)",
             "estimate": round(concordance(tte, ete, risk["Cox PH"]), 4),
             "ci_low": round(lo, 4), "ci_high": round(hi, 4), "se": round(se, 4),
             "method": "equipment cluster bootstrap", "n_resamples": B_BOOT})
print(f"{'Cox PH (clustered)':28s} 95% CI [{lo:.4f}, {hi:.4f}]  se {se:.4f}")

C_TARGET = 0.70
cox_lo = [r for r in rows if r["quantity"] == "concordance — Cox PH"][0]["ci_low"]
print(f"\nPre-registered criterion {C_TARGET}: point estimate "
      f"{'meets' if refit_cox >= C_TARGET else 'misses'}; lower confidence bound "
      f"{'also clears' if cox_lo >= C_TARGET else 'falls below'} the threshold.")

# Bootstrap intervals on Weibull parameter recovery

gt = spark.table(tbl("bronze_ground_truth_params")).toPandas().set_index("EQTYP")
for c in ["weibull_beta", "weibull_eta_days"]:
    gt[c] = gt[c].astype(float)

cov_cols = [c for c in Xtr.columns if not c.startswith("cls_") and c != "virtual_age"]
cov_idx = [Xtr.columns.get_loc(c) for c in cov_cols]
cov_coefs = coefs[cov_idx]
OFFSET_MEAN = float((Xtr.values[:, cov_idx] @ cov_coefs).mean())

def weibull_offset(t, e, va, offset):
    t, e = np.asarray(t, float), np.asarray(e, float)
    v, off = np.asarray(va, float), np.asarray(offset, float)
    def nll(p):
        b, eta = np.exp(p)
        ei = eta * np.exp(-off / b)
        x1, x0 = (v + t) / ei, v / ei
        logh = np.log(b / ei) + (b - 1) * np.log(np.maximum(x1, 1e-9))
        return -(np.sum(e * logh) - np.sum(x1 ** b - x0 ** b))
    r = minimize(nll, [np.log(1.8), np.log(max(t.mean(), 1) * 2)],
                 method="Nelder-Mead", options={"maxiter": 4000})
    return np.exp(r.x)

B_PARAM = 300
rec_rows = []
for cls, g in iv.groupby("equipment_class"):
    gtr = g[g.interval_start <= pd.Timestamp(TRAIN_CUTOFF)]
    if gtr.event_observed.sum() < 10:
        continue
    Xg = design(gtr).reindex(columns=Xtr.columns, fill_value=0.0)
    off = Xg.values[:, cov_idx] @ cov_coefs - OFFSET_MEAN
    t_, e_, v_ = (gtr.duration_days.values.astype(float),
                  gtr.event_observed.values.astype(float), gtr.va.values)
    b_hat, eta_hat = weibull_offset(t_, e_, v_, off)
    rs = np.random.default_rng(13); n = len(t_); bb, ee = [], []
    for _ in range(B_PARAM):
        idx = rs.integers(0, n, n)
        if e_[idx].sum() < 5:
            continue
        try:
            b2, e2 = weibull_offset(t_[idx], e_[idx], v_[idx], off[idx])
            bb.append(b2); ee.append(e2)
        except Exception:
            pass
    bt, et_ = float(gt.loc[cls, "weibull_beta"]), float(gt.loc[cls, "weibull_eta_days"])
    blo, bhi, _ = pct_ci(bb); elo, ehi, _ = pct_ci(ee)
    rec_rows.append({"equipment_class": cls, "train_failures": int(e_.sum()),
                     "beta_true": bt, "beta_fitted": round(b_hat, 3),
                     "beta_ci_low": round(blo, 3), "beta_ci_high": round(bhi, 3),
                     "beta_true_in_ci": bool(blo <= bt <= bhi),
                     "eta_true": et_, "eta_fitted": round(eta_hat, 1),
                     "eta_ci_low": round(elo, 1), "eta_ci_high": round(ehi, 1),
                     "eta_true_in_ci": bool(elo <= et_ <= ehi)})

rec_ci = pd.DataFrame(rec_rows)
display(spark.createDataFrame(rec_ci))
print(f"true beta inside 95% CI: {rec_ci.beta_true_in_ci.sum()}/{len(rec_ci)} classes")
print(f"true eta  inside 95% CI: {rec_ci.eta_true_in_ci.sum()}/{len(rec_ci)} classes")
print("Coverage near 95% indicates the estimator is honest about its own uncertainty even "
      "where the point estimate misses the 15% criterion.")

# Paired inference on the backtest deltas

# Regime comparisons share the same defect stream and the same fitted models within a
# replication, so the appropriate comparison is paired. Requires the per-replication totals
# written by notebook 12 — if absent, re-run 12 with the replication-level export enabled.

if spark.catalog.tableExists(tbl("results_backtest_replications")):
    rep = spark.table(tbl("results_backtest_replications")).toPandas()
    piv = rep.pivot(index="rep", columns="regime", values="total_cost")
    esc = rep.pivot(index="rep", columns="regime", values="escalation_cost")
    pairs = [("D full framework", "A baseline", piv, "RQ4 total-cost reduction"),
             ("B intervals only", "A baseline", piv, "RQ2 interval component"),
             ("C ordering only", "A baseline", piv, "RQ3 ordering component"),
             ("C ordering only", "A baseline", esc, "RQ3 escalation-cost reduction")]
    prs = []
    for x, base, frame, label in pairs:
        d = 100 * (frame[base] - frame[x]) / frame[base]
        lo, hi, se = pct_ci(d.values)
        w = stats.wilcoxon(frame[base], frame[x])
        prs.append({"quantity": label, "estimate": round(d.mean(), 2),
                    "ci_low": round(lo, 2), "ci_high": round(hi, 2),
                    "se": round(se, 2), "method": f"paired over {len(d)} replications",
                    "wilcoxon_p": float(f"{w.pvalue:.3g}")})
        print(f"{label:34s} {d.mean():6.2f}%  95% CI [{lo:.2f}, {hi:.2f}]  p={w.pvalue:.2e}")
    rows += prs
else:
    print("results_backtest_replications not found — add this to notebook 12 before the "
          "export block:\n"
          "    (spark.createDataFrame(res).write.mode('overwrite')\n"
          "         .option('overwriteSchema', True)\n"
          "         .saveAsTable(tbl('results_backtest_replications')))")

# Figure — estimates with confidence intervals

unc = pd.DataFrame(rows)
plot = unc[unc.quantity.str.startswith("concordance")].copy()
fig, ax = plt.subplots(figsize=(7.6, 3.2))
y = np.arange(len(plot))
ax.errorbar(plot.estimate, y,
            xerr=[plot.estimate - plot.ci_low, plot.ci_high - plot.estimate],
            fmt="o", color=ACCENT, ecolor=MUTED, capsize=3, markersize=6)
ax.axvline(C_TARGET, color=WARM, ls="--", lw=1.2)
ax.text(C_TARGET, len(plot) - 0.4, f" criterion {C_TARGET}", color=WARM, fontsize=8)
ax.set_yticks(y); ax.set_yticklabels([q.replace("concordance — ", "") for q in plot.quantity])
ax.set_xlabel("concordance with 95% bootstrap interval")
emit(fig, "u1_concordance_intervals")

# Persist

from pyspark.sql.functions import lit

(spark.createDataFrame(unc).withColumn("dataset_version", lit(DATASET_VERSION))
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(tbl("results_uncertainty")))
(spark.createDataFrame(rec_ci).withColumn("dataset_version", lit(DATASET_VERSION))
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(tbl("results_recovery_intervals")))
unc.to_csv(f"{EXPORTS}/results_uncertainty.csv", index=False)
rec_ci.to_csv(f"{EXPORTS}/results_recovery_intervals.csv", index=False)
print("written: results_uncertainty, results_recovery_intervals")